# Week 103, Day 1: Final exam: part 1

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w103d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('grading.py', 'w', encoding='utf-8') as f:
    f.write('def grade(score, out_of=100):\n    """The grade for a mark: distinction from 80%, merit from 70%, pass from 60%, otherwise fail."""\n    if out_of <= 0 or not 0 <= score <= out_of:\n        raise ValueError("score out of range")\n    percent = 100 * score / out_of\n    if percent >= 80:\n        return "distinction"\n    if percent >= 70:\n        return "merit"\n    if percent >= 60:\n        return "pass"\n    return "fail"\n')

*Interactive figure on the website: The timer keeps running if you reload or close the page. It doesn't stop you working when time is up: that's on your honour..*

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: B1: A bus timetable

A timetable has lines like `Mill: 07:15, 08:45, 07:50` (a stop, a colon, and times as `HH:MM` separated by commas); blank lines and lines starting with `#` are ignored. Write `timetable(text)`: a dict from stop to its times sorted, as strings; a stop listed twice has its times merged without repeats. Raise `ValueError("line N: bad time T")` for a time that isn't a valid 24-hour time (N counting every line from 1), and `ValueError("line N: no stop")` for a line without a colon.

In [ ]:
import re

def timetable(text):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_timetable():\n    \"strings, dictionaries, sorting\"\n    text = \"# Saturday buses\\nMill: 07:15, 08:45, 07:50\\n\\nRidge: 09:30\\nMill: 08:45, 06:05\\n\"\n    assert timetable(text) == {\"Mill\": [\"06:05\", \"07:15\", \"07:50\", \"08:45\"], \"Ridge\": [\"09:30\"]}\n    assert timetable(\"\") == {}\n\ndef test_errors():\n    \"clear errors\"\n    for text, message in [(\"Mill: 07:15\\nRidge: 24:00\", \"line 2: bad time 24:00\"), (\"Mill: 7:15\", \"line 1: bad time 7:15\"),\n                          (\"Mill 0715\", \"line 1: no stop\"), (\"Mill: 07:60\", \"line 1: bad time 07:60\")]:\n        try:\n            timetable(text)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)")

### Exercise: B2: Tests for grading

`grading.py` (on this page) has `grade(score, out_of=100)`. Write pytest tests that pass on it and catch all five broken versions.

In [ ]:
import pytest
from grading import grade

def test_top_mark():
    assert grade(100) == "distinction"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"grading.py\").read()\nBROKEN = {\n    \"distinction from 81\": GOOD.replace(\"percent >= 80\", \"percent > 80\"),\n    \"merit from 75\": GOOD.replace(\"percent >= 70\", \"percent >= 75\"),\n    \"pass from 50\": GOOD.replace(\"percent >= 60\", \"percent >= 50\"),\n    \"out_of ignored\": GOOD.replace(\"percent = 100 * score / out_of\", \"percent = score\"),\n    \"negative allowed\": GOOD.replace(\"not 0 <= score <= out_of\", \"score > out_of\"),\n}\n\ndef test_suite():\n    \"all five caught\"\n    for name, code in BROKEN.items():\n        assert code != GOOD, name\n    r = pytest_run(source)\n    assert r[\"failed\"] == [] and r[\"passed\"], r[\"output\"][-600:]\n    results = against(source, \"grading\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"grading.py\", GOOD)\n    missed = [k for k in BROKEN if not results[k]]\n    assert results[\"correct\"] == [] and not missed, f\"not caught: {missed}\"")

### Exercise: B3: Routes round closed paths

`paths` is a list of undirected `(a, b, km)`; `closed` is a set of paths closed today, each written as a `frozenset({a, b})`. Write `route_today(paths, closed, start, end)`: `(km, stops)` for the shortest open route (km rounded to 2 places; ties broken by the alphabetically smaller list of stops), or `None`.

In [ ]:
import heapq

def route_today(paths, closed, start, end):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("PATHS = [(\"Mill\", \"Weir\", 1.5), (\"Weir\", \"Church\", 2.0), (\"Mill\", \"Church\", 4.0), (\"Church\", \"Ridge\", 3.0),\n         (\"Weir\", \"Farm\", 2.5), (\"Farm\", \"Ridge\", 2.0), (\"Farm\", \"Wood\", 1.5), (\"Wood\", \"Lake\", 2.0), (\"Lake\", \"Mill\", 3.5)]\n\ndef test_routes():\n    \"Dijkstra, with some edges removed\"\n    assert route_today(PATHS, set(), \"Mill\", \"Ridge\") == (6.0, [\"Mill\", \"Weir\", \"Farm\", \"Ridge\"])\n    assert route_today(PATHS, {frozenset({\"Weir\", \"Farm\"})}, \"Mill\", \"Ridge\") == (6.5, [\"Mill\", \"Weir\", \"Church\", \"Ridge\"])\n    closed = {frozenset({\"Mill\", \"Weir\"}), frozenset({\"Mill\", \"Church\"}), frozenset({\"Lake\", \"Mill\"})}\n    assert route_today(PATHS, closed, \"Mill\", \"Ridge\") is None\n    assert route_today(PATHS, set(), \"Lake\", \"Lake\") == (0, [\"Lake\"])")

### Exercise: B4: Money that adds up

Write a value class `Money(amount, currency="GBP")`: `amount` is a string or `Decimal` (not a float: a float raises `TypeError`), kept to 2 decimal places (rounding half up). Money adds and subtracts with money of the same currency (otherwise `ValueError("currency mismatch")`), compares (`==`, `<` and the rest, via `functools.total_ordering`), hashes, and shows as `Money('12.50', 'GBP')`. `split(n)` returns `n` amounts that differ by at most a penny and add up exactly, the larger ones first.

In [ ]:
import functools
from decimal import Decimal, ROUND_HALF_UP

@functools.total_ordering
class Money:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from decimal import Decimal\n\ndef test_money():\n    \"a value class\"\n    a, b = Money(\"10.005\"), Money(Decimal(\"2.5\"))\n    assert repr(a) == \"Money('10.01', 'GBP')\" and a + b == Money(\"12.51\") and a - b == Money(\"7.51\")\n    assert Money(\"1\") < Money(\"1.01\") and Money(\"2\") >= Money(\"2.00\") and len({Money(\"3\"), Money(\"3.00\")}) == 1\n    assert Money(\"12.51\").split(3) == [Money(\"4.17\"), Money(\"4.17\"), Money(\"4.17\")]\n    assert Money(\"10.00\").split(3) == [Money(\"3.34\"), Money(\"3.33\"), Money(\"3.33\")]\n    assert sum((m for m in Money(\"0.05\").split(4)), Money(\"0\")) == Money(\"0.05\")\n    for action, kind in [(lambda: Money(1.5), TypeError), (lambda: Money(\"1\") + Money(\"1\", \"EUR\"), ValueError)]:\n        try:\n            action()\n        except kind:\n            pass\n        else:\n            raise AssertionError(kind)")

### Exercise: B5: Checking argument types

Write a decorator `checked` that reads the decorated function's parameter annotations with `inspect.signature` and, on each call, raises `TypeError("PARAM must be TYPE")` (TYPE the annotation's `__name__`) when an argument annotated `int`, `float`, `str` or `bool` has the wrong type. A `bool` doesn't count as an `int` or `float`; an `int` does count as a `float`. Unannotated parameters and other annotations aren't checked; defaults apply. Keep the function's name.

In [ ]:
import functools
import inspect

def checked(fn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_checked():\n    \"inspect and decorators\"\n    @checked\n    def plan(start: str, km: float, stops: int = 1, note=None):\n        return f\"{start} {km} {stops}\"\n    assert plan(\"Mill\", 6, 2) == \"Mill 6 2\" and plan(\"Mill\", 6.5) == \"Mill 6.5 1\" and plan(start=\"Mill\", km=1.0, note=[1]) == \"Mill 1.0 1\"\n    for args, kwargs, message in [((\"Mill\", \"6\"), {}, \"km must be float\"), ((\"Mill\", True), {}, \"km must be float\"),\n                                  ((7, 6.0), {}, \"start must be str\"), ((\"Mill\", 6.0, 2.5), {}, \"stops must be int\"),\n                                  ((\"Mill\", 6.0), {\"stops\": False}, \"stops must be int\")]:\n        try:\n            plan(*args, **kwargs)\n        except TypeError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)\n    assert plan.__name__ == \"plan\"")

### Exercise: B6: An access log summary

Each line of a web server's access log reads `METHOD PATH STATUS MILLISECONDS` (for example `GET /v1/walks 200 34`); lines that don't match are counted as malformed. Write `log_summary(lines)`: a dict with `requests` (well-formed lines), `malformed`, `by_class` (counts by status class such as `"2xx"`, sorted by class), `error_rate` (the percentage of 5xx responses, 1 decimal place), `p95_ms` (nearest-rank 95th percentile of the times, or `None`), and `top_paths` (the three most requested paths, ignoring query strings, most first, ties alphabetical).

In [ ]:
import math
import re
from collections import Counter

def log_summary(lines):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("LOG = [\"GET /v1/walks 200 34\", \"GET /v1/walks?limit=5 200 41\", \"POST /v1/walks 201 88\", \"GET /v1/walks/7 404 12\",\n       \"GET /v1/plan 500 950\", \"GET /v1/plan 200 120\", \"garbage\", \"DELETE /v1/walks/7 204 20\", \"GET /v1/walks 200 30\",\n       \"GET /v1/plan 503 1200\", \"GET  broken\"]\n\ndef test_summary():\n    \"parsing, counting, percentiles\"\n    s = log_summary(LOG)\n    assert s[\"requests\"] == 9 and s[\"malformed\"] == 2\n    assert s[\"by_class\"] == {\"2xx\": 6, \"4xx\": 1, \"5xx\": 2} and list(s[\"by_class\"]) == [\"2xx\", \"4xx\", \"5xx\"]\n    assert s[\"error_rate\"] == 22.2 and s[\"p95_ms\"] == 1200 and s[\"top_paths\"] == [\"/v1/walks\", \"/v1/plan\", \"/v1/walks/7\"]\n    assert log_summary([]) == {\"requests\": 0, \"malformed\": 0, \"by_class\": {}, \"error_rate\": 0.0, \"p95_ms\": None, \"top_paths\": []}")

## Before you start

**The rules.** You have **2 hours and 30 minutes**. Work alone; close your notes, don't use the tutor, and don't search the web. You may run your code and press **Check my answer** as often as you like.

**The marks.**

| Part | What | Marks |
|---|---|---|
| A | 15 quiz questions | 15 (1 each) |
| B | 6 problems | 90 (15 each) |
| | **Part 1 total** | **105** |

Part 2, tomorrow, is worth 75, so the exam is out of **180**: **108 (60%)** passes, **144 (80%)** is a distinction. Count a problem's marks only when every check passes.

**Strategy.** Read all six problems first and note which tool each needs. Use your time budget from Week 102 Day 6, and don't stay stuck on one problem past its share: come back to it.

## Part A: quiz

## Part B: six problems

## When you finish

Add up your marks: 1 for each quiz question right first time, and 15 for each problem whose checks all pass. Write the total down with the time you took, and a note of anything you didn't finish. Don't look anything up and don't revise tonight: part 2 tomorrow asks you to build one larger program and review someone else's code. Rest.